# 🧪 Notebook 11: Robust and Resampling Inference

> Run with `jupyter nbconvert --to notebook --execute <this notebook>` (or open it in JupyterLab) to populate outputs. Every random step is seeded, so results are reproducible.

## Aims

Default `statsmodels` output reports *asymptotic, model-based* standard errors that assume
homoskedastic, independent errors. This notebook shows how inference changes when those
assumptions fail, and how resampling gives assumption-light alternatives:

1. Heteroskedasticity-consistent (HC0–HC3), HAC and cluster-robust standard errors
2. Bootstrap confidence intervals (percentile, basic, normal, BCa) for a skewed statistic
3. Bootstrap standard errors for regression coefficients (pairs, residual, wild) vs. HC3
4. Permutation tests vs. the t-test
5. Effect sizes with confidence intervals
6. Multiple-comparison corrections

Datasets: `heteroskedastic_data.csv` (DGP: `y = 3 + 2X + e`, `e ~ N(0, (1 + 2|X|)^2)`) and
`posthoc_dataset.csv` (three groups with means 60, 70, 65).

In [ ]:
# Setup and imports
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.inference import (
    robust_se_table, bootstrap_ci, bootstrap_regression,
    permutation_test, multiple_testing,
)
from utils.effect_sizes import cohens_d, hedges_g, cliffs_delta, anova_effect_sizes
from utils.reporting import save_table
warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 110

DATA_PATH = Path("../synthetic_data")
EXPORT_PLOTS = Path("../exports/plots/11_robust_and_resampling_inference")
EXPORT_TABLES = Path("../exports/tables/11_robust_and_resampling_inference")
EXPORT_PLOTS.mkdir(parents=True, exist_ok=True)
EXPORT_TABLES.mkdir(parents=True, exist_ok=True)

SEED = 2026

In [ ]:
het = pd.read_csv(DATA_PATH / "heteroskedastic_data.csv")
post = pd.read_csv(DATA_PATH / "posthoc_dataset.csv")
print(het.describe().T)
print(post.groupby("Group")["Score"].agg(["count", "mean", "std"]))

## 1. Robust standard errors

The true slope is 2. Under heteroskedasticity OLS coefficients remain unbiased, but the
conventional SE is wrong. `robust_se_table` refits the model under several covariance
estimators. Cluster-robust SEs are shown with *artificial* clusters (blocks of 10 rows)
purely to illustrate the mechanics.

In [ ]:
ols = smf.ols("y ~ X", data=het).fit()
clusters = np.repeat(np.arange(len(het) // 10), 10)
se_table = robust_se_table(
    ols, cov_types=("nonrobust", "HC0", "HC1", "HC2", "HC3", "HAC", "cluster"), groups=clusters
)
se_wide = se_table.pivot(index="term", columns="cov_type", values="se")
se_wide = se_wide[["nonrobust", "HC0", "HC1", "HC2", "HC3", "HAC", "cluster"]]
se_wide

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
se_wide.loc["X"].plot(kind="bar", ax=ax, color="steelblue")
ax.axhline(se_wide.loc["X", "nonrobust"], color="red", ls="--", label="nonrobust")
ax.set_ylabel("SE of slope on X")
ax.set_title("Standard error of the slope under different covariance estimators")
ax.legend()
plt.tight_layout()
plt.savefig(EXPORT_PLOTS / "robust_se_comparison.png")
plt.show()
se_wide.to_csv(EXPORT_TABLES / "robust_se_table.csv")

The HC estimators are roughly 25–50% larger than the conventional SE here, so the
conventional interval is too narrow. HC3 is the most conservative and is the usual
recommendation for small samples (Long & Ervin, 2000).

## 2. Bootstrap confidence intervals for a skewed statistic

For a right-skewed statistic (the mean of a log-normal sample, n = 40), the sampling
distribution is asymmetric. The percentile interval ignores bias and skew; the BCa
interval corrects for both (DiCiccio & Efron, 1996).

In [ ]:
rng = np.random.default_rng(SEED)
skewed = rng.lognormal(mean=0.0, sigma=1.0, size=40)
true_mean = np.exp(0.5)  # population mean of LogNormal(0, 1)

boot_rows = []
for method in ["percentile", "basic", "normal", "bca"]:
    res = bootstrap_ci(skewed, np.mean, n_boot=1000, alpha=0.05, method=method, seed=SEED)
    boot_rows.append({"method": method, "estimate": res.estimate, "ci_low": res.ci_low,
                      "ci_high": res.ci_high, "width": res.ci_high - res.ci_low, "boot_se": res.se,
                      "covers_truth": res.ci_low <= true_mean <= res.ci_high})
boot_table = pd.DataFrame(boot_rows).set_index("method")
print(f"Population mean = {true_mean:.3f}; sample mean = {skewed.mean():.3f}")
boot_table

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
for i, (m, row) in enumerate(boot_table.iterrows()):
    ax.plot([row.ci_low, row.ci_high], [i, i], lw=3, label=m)
    ax.plot(row.estimate, i, "ko")
ax.axvline(true_mean, color="red", ls="--", label="population mean")
ax.set_yticks(range(len(boot_table)))
ax.set_yticklabels(boot_table.index)
ax.set_xlabel("95% CI for the mean")
ax.set_title("Bootstrap CI methods for a skewed statistic (n = 40, B = 1000)")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.savefig(EXPORT_PLOTS / "bootstrap_ci_methods.png")
plt.show()
boot_table.to_csv(EXPORT_TABLES / "bootstrap_ci_methods.csv")

## 3. Bootstrap standard errors for regression coefficients

- **Pairs** bootstrap resamples rows: robust to heteroskedasticity and misspecified error distributions.
- **Residual** bootstrap resamples residuals: assumes i.i.d. errors, so it should *undercover* here.
- **Wild** bootstrap multiplies leverage-adjusted residuals by Rademacher signs: robust to heteroskedasticity while keeping the design fixed.

Compare all three with the HC3 asymptotic SE.

In [ ]:
reg_rows = []
for method in ["pairs", "residual", "wild"]:
    t = bootstrap_regression("y ~ X", het, n_boot=1000, method=method, seed=SEED)
    reg_rows.append({"method": method, "boot_se_X": t.loc["X", "boot_se"],
                     "ci_low": t.loc["X", "ci_low"], "ci_high": t.loc["X", "ci_high"]})
hc3_se = smf.ols("y ~ X", data=het).fit(cov_type="HC3").bse["X"]
reg_rows.append({"method": "HC3 (asymptotic)", "boot_se_X": hc3_se, "ci_low": np.nan, "ci_high": np.nan})
reg_rows.append({"method": "nonrobust (asymptotic)", "boot_se_X": ols.bse["X"], "ci_low": np.nan, "ci_high": np.nan})
reg_table = pd.DataFrame(reg_rows).set_index("method")
reg_table.to_csv(EXPORT_TABLES / "bootstrap_regression_se.csv")
reg_table

Pairs and wild bootstrap SEs agree closely with HC3, while the residual bootstrap
reproduces the (too small) conventional SE because it imposes homoskedasticity.

## 4. Permutation test vs. t-test

Group B (mean 70) vs. group C (mean 65) in `posthoc_dataset.csv`. The permutation test
needs only exchangeability under the null, not normality.

In [ ]:
b = post.loc[post.Group == "B", "Score"].to_numpy()
c = post.loc[post.Group == "C", "Score"].to_numpy()
perm = permutation_test(b, c, statistic="mean_diff", n_perm=5000, seed=SEED)
welch = stats.ttest_ind(b, c, equal_var=False)
print(f"Observed mean difference: {perm.observed:.3f}")
print(f"Permutation p-value: {perm.p_value:.4f}   Welch t-test p-value: {welch.pvalue:.4g}")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(perm.null_distribution, bins=50, color="lightgray", edgecolor="white")
ax.axvline(perm.observed, color="red", lw=2, label="observed")
ax.axvline(-perm.observed, color="red", lw=1, ls=":")
ax.set_title("Permutation null distribution of the mean difference (B - C)")
ax.set_xlabel("mean difference")
ax.legend()
plt.tight_layout()
plt.savefig(EXPORT_PLOTS / "permutation_null.png")
plt.show()

## 5. Effect sizes with confidence intervals

A p-value says nothing about magnitude. Standardised mean differences and ordinal effect
sizes are reported with 95% intervals.

In [ ]:
d = cohens_d(b, c)
g = hedges_g(b, c)
cd = cliffs_delta(b, c)
effects = pd.DataFrame([
    {"effect_size": "Cohen's d", "value": d["d"], "ci_low": d["ci_low"], "ci_high": d["ci_high"], "label": d["interpretation"]},
    {"effect_size": "Hedges' g", "value": g["g"], "ci_low": g["ci_low"], "ci_high": g["ci_high"], "label": g["interpretation"]},
    {"effect_size": "Cliff's delta", "value": cd["delta"], "ci_low": np.nan, "ci_high": np.nan, "label": cd["interpretation"]},
]).set_index("effect_size")

anova = sm.stats.anova_lm(smf.ols("Score ~ C(Group)", data=post).fit(), typ=2)
print(anova_effect_sizes(anova))
effects

## 6. Multiple comparisons

All three pairwise comparisons are tested with permutation tests, then adjusted with
Holm (family-wise error) and Benjamini–Hochberg (false discovery rate).

In [ ]:
groups = {k: v["Score"].to_numpy() for k, v in post.groupby("Group")}
pairs, pvals = [], []
for i, gi in enumerate(groups):
    for gj in list(groups)[i + 1:]:
        r = permutation_test(groups[gi], groups[gj], n_perm=3000, seed=SEED)
        pairs.append(f"{gi} vs {gj}")
        pvals.append(r.p_value)

holm = multiple_testing(pvals, method="holm", labels=pairs)
bh = multiple_testing(pvals, method="fdr_bh", labels=pairs)
mt = holm.rename(columns={"p_adj": "p_holm", "reject": "reject_holm"}).drop(columns="method")
mt["p_bh"] = bh["p_adj"]
mt["reject_bh"] = bh["reject"]
mt

In [ ]:
# Summary table for export
summary = pd.concat([
    se_wide.loc[["X"]].rename(index={"X": "SE of slope (heteroskedastic data)"}),
    reg_table[["boot_se_X"]].T.rename(index={"boot_se_X": "Bootstrap SE of slope"}),
], axis=0)
save_table(summary, EXPORT_TABLES / "inference_summary", formats=("csv", "md"))
save_table(mt, EXPORT_TABLES / "pairwise_multiple_testing", formats=("csv", "md"))
save_table(effects, EXPORT_TABLES / "effect_sizes_B_vs_C", formats=("csv", "md"))
print("Saved tables to", EXPORT_TABLES)

## References

- White, H. (1980). A heteroskedasticity-consistent covariance matrix estimator and a direct test for heteroskedasticity. *Econometrica*, 48(4), 817–838.
- MacKinnon, J. G., & White, H. (1985). Some heteroskedasticity-consistent covariance matrix estimators with improved finite sample properties. *Journal of Econometrics*, 29(3), 305–325.
- Long, J. S., & Ervin, L. H. (2000). Using heteroscedasticity consistent standard errors in the linear regression model. *The American Statistician*, 54(3), 217–224.
- Newey, W. K., & West, K. D. (1987). A simple, positive semi-definite, heteroskedasticity and autocorrelation consistent covariance matrix. *Econometrica*, 55(3), 703–708.
- Efron, B., & Tibshirani, R. J. (1993). *An Introduction to the Bootstrap*. Chapman & Hall.
- DiCiccio, T. J., & Efron, B. (1996). Bootstrap confidence intervals. *Statistical Science*, 11(3), 189–228.
- Wu, C. F. J. (1986). Jackknife, bootstrap and other resampling methods in regression analysis. *Annals of Statistics*, 14(4), 1261–1295.
- Phipson, B., & Smyth, G. K. (2010). Permutation p-values should never be zero. *Statistical Applications in Genetics and Molecular Biology*, 9(1).
- Lakens, D. (2013). Calculating and reporting effect sizes to facilitate cumulative science. *Frontiers in Psychology*, 4, 863.
- Holm, S. (1979). A simple sequentially rejective multiple test procedure. *Scandinavian Journal of Statistics*, 6(2), 65–70.
- Benjamini, Y., & Hochberg, Y. (1995). Controlling the false discovery rate. *JRSS-B*, 57(1), 289–300.